# 🎯 Technique 79: Gradient-Free Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/79_gradient_free_optimization.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  
**Technique #:** 79  
**Difficulty:** Advanced

## 📋 Description

Gradient-Free Optimization (GFO) refers to optimization methods that do not require gradient information. For prompt engineering, this means searching for optimal prompts without differentiable access to the model.

**When to use:**
- When you cannot access model gradients
- For discrete optimization problems
- Black-box optimization scenarios

## 🔧 How It Works

GFO methods include:
- Evolutionary Strategies
- Bayesian Optimization
- Simulated Annealing

These methods explore the search space without requiring gradients.

## ⚙️ Setup

In [ ]:
!pip install -q openai numpy
import openai
import numpy as np
import random
from typing import List, Tuple
from dataclasses import dataclass
from getpass import getpass

In [ ]:
openai.api_key = getpass('Enter your OpenAI API key: ')

## 🛠️ Implementation: Evolutionary Optimizer

In [ ]:
@dataclass
class Individual:
    instruction: str
    temperature: float
    top_p: float
    fitness: float = 0.0

class EvoOptimizer:
    def __init__(self, model='gpt-4o-mini', pop_size=10):
        self.model = model
        self.pop_size = pop_size
        self.population = []
    
    def call_llm(self, prompt, temp=0.7, top_p=1.0):
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temp,
            top_p=top_p
        )
        return response.choices[0].message.content.strip()
    
    def init_population(self, base_instruction):
        for _ in range(self.pop_size):
            self.population.append(Individual(
                instruction=base_instruction,
                temperature=random.uniform(0, 1),
                top_p=random.uniform(0.5, 1)
            ))
    
    def evaluate(self, individual, test_cases, task_desc):
        correct = 0
        for test_input, expected in test_cases:
            prompt = f'{individual.instruction}\n{task_desc}\nInput: {test_input}\nOutput:'
            try:
                response = self.call_llm(prompt, individual.temperature, individual.top_p)
                if expected.lower() in response.lower():
                    correct += 1
            except:
                pass
        individual.fitness = correct / len(test_cases)
        return individual.fitness
    
    def mutate(self, individual):
        mutant = Individual(
            instruction=individual.instruction + ' Be concise.',
            temperature=np.clip(individual.temperature + random.gauss(0, 0.1), 0, 1),
            top_p=np.clip(individual.top_p + random.gauss(0, 0.1), 0.1, 1)
        )
        return mutant
    
    def evolve(self, test_cases, task_desc, generations=3):
        for gen in range(generations):
            for ind in self.population:
                self.evaluate(ind, test_cases, task_desc)
            self.population.sort(key=lambda x: x.fitness, reverse=True)
            print(f'Gen {gen+1}: Best fitness = {self.population[0].fitness:.2%}')
            # Keep best, mutate rest
            new_pop = [self.population[0]]
            for _ in range(self.pop_size - 1):
                parent = random.choice(self.population[:3])
                new_pop.append(self.mutate(parent))
            self.population = new_pop
        return self.population[0]

## 💡 Basic Example: Optimizing Classification

In [ ]:
test_cases = [
    ('I love this!', 'POSITIVE'),
    ('This is terrible.', 'NEGATIVE'),
    ('It was okay.', 'NEUTRAL')
]

optimizer = EvoOptimizer(pop_size=5)
optimizer.init_population('Analyze sentiment.')

best = optimizer.evolve(test_cases, 'Classify as POSITIVE, NEGATIVE, or NEUTRAL.', generations=3)

print(f'\nBest config: temp={best.temperature:.2f}, top_p={best.top_p:.2f}')
print(f'Instruction: {best.instruction}')

## 🌍 Real-World Example: Multi-Parameter Optimization

In [ ]:
content_cases = [
    ('Write a product description for wireless earbuds', 'earbuds wireless'),
    ('Create a social media post about coffee', 'coffee social'),
]

def eval_content(prompt, response, keywords):
    score = 0
    for kw in keywords.split():
        if kw in response.lower():
            score += 0.5
    return min(score, 1.0)

print('Content optimization example - customize eval_content for your needs')

## ⚠️ Failure Case: GFO Limitations

In [ ]:
print('GFO Limitations:\n')
print('1. Requires many evaluations (expensive)')
print('2. Can get stuck in local optima')
print('3. Noisy evaluation from LLM stochasticity')
print('4. Struggles with high-dimensional spaces')
print('5. Discrete optimization is challenging')

## 📊 GFO Performance Comparison

| Method | Convergence | Quality | API Calls |
|--------|-------------|---------|-----------|\n| Random Search | Slow | Low | Many |
| Evolutionary | Medium | High | Many |
| Bayesian Opt | Fast | High | Few |

## 🎮 Interactive Playground

In [ ]:
YOUR_TASK = 'Your task description'
YOUR_TEST_CASES = [('input1', 'expected1'), ('input2', 'expected2')]
YOUR_BASE_INSTRUCTION = 'Your base instruction'

# my_opt = EvoOptimizer(pop_size=5)
# my_opt.init_population(YOUR_BASE_INSTRUCTION)
# best = my_opt.evolve(YOUR_TEST_CASES, YOUR_TASK, generations=3)

## 💡 Tips & Tricks

- Use Bayesian Opt for continuous parameters
- Use Evolutionary for discrete spaces
- Population size: 10-50
- Mutation rate: 0.1-0.3
- Always use elitism

## 📚 References

1. [CMA-ES Tutorial](https://arxiv.org/abs/1604.00772)
2. [Bayesian Optimization Primer](https://arxiv.org/abs/1807.02811)